In [2]:
# Install the Kaggle library
!pip install -q kaggle

# Prompt to upload the kaggle.json file
from google.colab import files
print("Please upload your kaggle.json file:")
uploaded = files.upload()

# Set up the Kaggle directory and permissions
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download the M5 Forecasting Accuracy dataset
print("Downloading M5 dataset from Kaggle...")
!kaggle competitions download -c m5-forecasting-accuracy

# Create a directory and unzip the files silently
print("Unzipping files...")
!mkdir -p m5_data
!unzip -q m5-forecasting-accuracy.zip -d m5_data
print("M5 Data successfully downloaded and extracted!")

Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json
100% 45.8M/45.8M [00:03<00:00, 12.1MB/s]

Unzipping files...
M5 Data successfully downloaded and extracted!


In [3]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

def load_and_preprocess_m5(data_dir, store_id='CA_1', lookback=168, horizon=24):
    print("Loading raw M5 datasets...")
    sales = pd.read_csv(f"{data_dir}/sales_train_evaluation.csv")
    calendar = pd.read_csv(f"{data_dir}/calendar.csv")

    print(f"Aggregating hierarchical data for warehouse: {store_id}...")
    # Isolate the specific 3PL facility
    store_sales = sales[sales['store_id'] == store_id]

    # Aggregate all item-level sales to calculate total daily floor throughput
    daily_volume = store_sales.loc[:, 'd_1':].sum(axis=0).values

    print("Merging multivariate calendar features...")
    cal_subset = calendar.iloc[:len(daily_volume)].copy()

    # Feature Engineering
    cal_subset['volume'] = daily_volume
    # Convert categorical events to binary indicators
    cal_subset['is_event'] = cal_subset['event_name_1'].notna().astype(int)
    cal_subset['is_weekend'] = cal_subset['wday'].apply(lambda x: 1 if x <= 2 else 0)

    # Final Multivariate Array: [Volume, Weekend, Event, SNAP_CA]
    features = cal_subset[['volume', 'is_weekend', 'is_event', 'snap_CA']].values

    print("Normalizing features...")
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_features = scaler.fit_transform(features)

    print("Generating temporal sliding windows...")
    X, y = [], []
    for i in range(len(scaled_features) - lookback - horizon + 1):
        X.append(scaled_features[i : i + lookback])
        # The target prediction is only the volume (index 0)
        y.append(scaled_features[i + lookback : i + lookback + horizon, 0])

    X, y = np.array(X), np.array(y)

    # Chronological Split (No Shuffling to prevent data leakage)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, shuffle=False)

    print("\n--- Data Pipeline Complete ---")
    print(f"Training Tensor Shape: {X_train.shape}")
    print(f"Testing Tensor Shape: {X_test.shape}")

    return (torch.tensor(X_train, dtype=torch.float32),
            torch.tensor(y_train, dtype=torch.float32),
            torch.tensor(X_test, dtype=torch.float32),
            torch.tensor(y_test, dtype=torch.float32),
            scaler)

# Execute the pipeline
X_train, y_train, X_test, y_test, scaler = load_and_preprocess_m5('m5_data')

Loading raw M5 datasets...
Aggregating hierarchical data for warehouse: CA_1...
Merging multivariate calendar features...
Normalizing features...
Generating temporal sliding windows...

--- Data Pipeline Complete ---
Training Tensor Shape: (1487, 168, 4)
Testing Tensor Shape: (263, 168, 4)


In [4]:
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(0), :]

class ShiftTransformer(nn.Module):
    # Notice input_dim is now explicitly set to 4
    def __init__(self, input_dim=4, d_model=128, nhead=8, num_layers=3, dropout=0.1):
        super(ShiftTransformer, self).__init__()
        self.model_type = 'Transformer'

        # Maps the 4 input features into the 128-dimensional embedding space
        self.encoder_layer = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layers = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward=512,
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layers, num_layers
        )

        # Output mapping back to a single predicted volume metric
        self.decoder = nn.Linear(d_model, 1)

    def forward(self, src):
        src = self.encoder_layer(src)
        src = self.pos_encoder(src)
        output = self.transformer_encoder(src)
        output = self.decoder(output)
        return output

# Initialize the environment
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

# Instantiate the updated model
model = ShiftTransformer(input_dim=4).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

print("Multivariate Transformer architecture successfully initialized!")

Using compute device: cuda
Multivariate Transformer architecture successfully initialized!


In [5]:
import time
from torch.utils.data import TensorDataset, DataLoader

# 1. Create DataLoaders for efficient batching
batch_size = 64
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

epochs = 50
train_losses = []
horizon = 24 # The 24-hour prediction window

print(f"Starting Model Training on {device}...")
for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    start_time = time.time()

    for batch_X, batch_y in train_loader:
        # Move tensors to GPU (if available)
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(batch_X)

        # The model outputs a sequence of 168, we slice the last 24 steps for the forecast horizon
        predictions = outputs[:, -horizon:, 0]

        # Calculate loss and backpropagate
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    elapsed = time.time() - start_time

    # Print progress every 5 epochs
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d}/{epochs} | Loss (MSE): {avg_loss:.5f} | Time: {elapsed:.2f}s")

print("\n--- Training Complete! ---")

Starting Model Training on cuda...
Epoch 01/50 | Loss (MSE): 0.04921 | Time: 1.75s
Epoch 05/50 | Loss (MSE): 0.02045 | Time: 0.85s
Epoch 10/50 | Loss (MSE): 0.01686 | Time: 0.85s
Epoch 15/50 | Loss (MSE): 0.01522 | Time: 0.86s
Epoch 20/50 | Loss (MSE): 0.01426 | Time: 0.87s
Epoch 25/50 | Loss (MSE): 0.01329 | Time: 0.87s
Epoch 30/50 | Loss (MSE): 0.01315 | Time: 0.87s
Epoch 35/50 | Loss (MSE): 0.01267 | Time: 0.89s
Epoch 40/50 | Loss (MSE): 0.01245 | Time: 0.89s
Epoch 45/50 | Loss (MSE): 0.01227 | Time: 0.89s
Epoch 50/50 | Loss (MSE): 0.01214 | Time: 0.89s

--- Training Complete! ---


In [6]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("Evaluating model on out-of-sample M5 test data...")

# Set model to evaluation mode
model.eval()

with torch.no_grad():
    # Push test data to GPU and get predictions
    test_outputs = model(X_test.to(device))
    # Slice the 24-hour horizon (index 0 is the volume feature)
    real_predictions = test_outputs[:, -horizon:, 0].cpu().numpy()

# Extract actual ground truth values
real_y = y_test.numpy()

# Inverse transform to get actual unit volumes
# We create dummy arrays because the scaler expects 4 columns
dummy_pred = np.zeros((real_predictions.flatten().shape[0], 4))
dummy_pred[:, 0] = real_predictions.flatten()
inv_predictions = scaler.inverse_transform(dummy_pred)[:, 0].reshape(real_predictions.shape)

dummy_y = np.zeros((real_y.flatten().shape[0], 4))
dummy_y[:, 0] = real_y.flatten()
inv_y = scaler.inverse_transform(dummy_y)[:, 0].reshape(real_y.shape)

# Calculate final metrics
mae = mean_absolute_error(inv_y, inv_predictions)
rmse = np.sqrt(mean_squared_error(inv_y, inv_predictions))

print(f"\n--- Final Model Performance ---")
print(f"Mean Absolute Error (MAE): {mae:.2f} Units")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} Units")

Evaluating model on out-of-sample M5 test data...

--- Final Model Performance ---
Mean Absolute Error (MAE): 1281.11 Units
Root Mean Squared Error (RMSE): 1659.34 Units
